# 🦜🔗 LangChain Study Notes: Tools & Function Calling

## 📌 Quick Reference & Overview
LLMs alone cannot access live data or perform real-world actions. **Tools** bridge this gap by allowing LLMs to request execution of Python functions (e.g. fetching weather, querying database, web searching).

### 🔑 Key Concepts Covered:
1. **The `@tool` Decorator**: Converts standard Python functions into LangChain `BaseTool` objects.
2. **Binding Tools (`bind_tools`)**: Attaches tool schemas to a model without forcing execution.
3. **Tool Call Payloads (`response.tool_calls`)**: Extracting target function names and JSON arguments generated by the model.
4. **Manual Tool Execution Loop**: The 3-step loop (Prompt -> Model tool call -> Function execution -> Final answer synthesis).

---

### 🛠️ Step 1: Base Model Initialization
Initialize LLM (e.g. Groq `qwen/qwen3.8-27b`) and test a standard text prompt before binding tools.

In [2]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3.8-27b")
response = model.invoke("Why do parrot talk?")
response

AIMessage(content='Actually, parrots don’t "talk" in the human sense—they **mimic sounds**. This ability is known as **vocal mimicry**. Here’s why and how they do it:\n\n### 1. **Social Bonding and Communication**\nParrots are highly social, flocking animals. In the wild, they use a wide variety of calls to:\n- Stay in contact with their flock.\n- Warn of predators.\n- Mark territory.\n- Strengthen pair bonds.\n\nIn captivity, humans become their "flock." By mimicking human speech, they:\n- Seek attention and interaction.\n- Reinforce their bond with their owners.\n- Feel part of their "social group."\n\n### 2. **Biological Advantage**\nParrots have a unique vocal organ called the **syrinx**, which allows them to produce a wide range of sounds. Some species (like African Greys, Amazons, and Cockatiels) have particularly complex syrinx structures that enable them to replicate human speech with surprising accuracy.\n\n### 3. **Cognitive Ability and Motivation**\n- **Intelligence**: Parro

---
### ⚙️ Step 2: Tool Creation (`@tool`) & Model Binding (`bind_tools`)

#### 1️⃣ The `@tool` Decorator
- Converts `get_weather(location: str) -> str` into a tool.
- **Docstring**: Function docstrings (`"""Get the weather at a location"""`) serve as the description read by the LLM.
- **Type Hints**: Parameter types (`location: str`) tell the LLM what data types to generate.

#### 2️⃣ `model.bind_tools([get_weather])`
- Passes the JSON schema of `get_weather` to the provider API.

In [3]:
from langchain.tools import tool

@tool
def get_weather(location: str)->str:
    """Get the weather at a location """
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])

---
### 🔍 Step 3: Invoking Model with Tools & Inspecting `tool_calls` 

- When asked *"What's the weather like in Boston?"*, the LLM realizes it needs external data.
- Instead of generating text, it returns an `AIMessage` with `finish_reason='tool_calls'`.
- `response.tool_calls` contains a list of tool call dicts: `[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': '...'}]`.

In [6]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    print(f"Tool: {tool_call["name"]}")
    print(f"Args: {tool_call["args"]}")

content='' additional_kwargs={'tool_calls': [{'id': 'p5gd59fky', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 26, 'prompt_tokens': 278, 'total_tokens': 304, 'completion_time': 0.067929285, 'completion_tokens_details': None, 'prompt_time': 0.019455805, 'prompt_tokens_details': None, 'queue_time': 0.050008005, 'total_time': 0.08738509}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_424cb89518', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--01a0bd23-8cd2-7702-99df-ce32bfdde16c-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'p5gd59fky', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 278, 'output_tokens': 26, 'total_tokens': 304}
Tool: get_weather
Args: {'location': 'Boston'}


---
### 🔁 Step 4: The 3-Step Manual Tool Execution Loop

#### Execution Sequence:
1. **Step 1 (Model Request)**: Send prompt to `model_with_tools`. Model responds with `tool_calls` payload. Append `ai_msg` to `messages` history.
2. **Step 2 (Tool Execution)**: Iterate through `ai_msg.tool_calls`, execute `get_weather.invoke(tool_call)`. This produces a `ToolMessage` containing the function output. Append result to `messages`.
3. **Step 3 (Final Answer Synthesis)**: Pass updated `messages` history back to `model_with_tools`. The model reads the `ToolMessage` and answers the user in natural language.

In [7]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content":"What's the weather like in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool call with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)
# "The current weather in Boston is 72*F and sunny."

It's sunny in Boston! ☀️
